In [1]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22

# =========================
# LOAD BOTH SHEETS
# =========================

hz = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="HZ")
vt = pd.read_excel("C:/Users/Ex0164/Book1.xlsx", sheet_name="VT")

# =========================
# FUNCTION TO MELT MACHINES
# =========================

def expand_machines(df):

    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_long = df.melt(
        id_vars=["Part","Inventory","Indent","Category","Cycle time","Cavity"],
        value_vars=machine_cols,
        var_name="Machine_Col",
        value_name="Machine"
    )

    df_long = df_long.dropna(subset=["Machine"])

    return df_long

hz_long = expand_machines(hz)
vt_long = expand_machines(vt)

df = pd.concat([hz_long, vt_long], ignore_index=True)

# =========================
# CALCULATE RATE
# =========================

df["Rate"] = (3600 / df["Cycle time"]) * df["Cavity"]

# =========================
# MRP LOGIC
# =========================

df["Daily_Demand"] = df["Monthly_Indent"] / WORKING_DAYS
df["Safety"] = df["Daily_Demand"] * 3
df["Gap"] = df["Safety"] - df["Inventory"]
df["Trigger"] = np.where(df["Gap"] > 0, 1, 0)

# =========================
# CATEGORY TARGET
# =========================

def target_days(cat):
    if cat == "Runner":
        return 5
    elif cat == "Repeater":
        return 3
    else:
        return 1

df["Target_Days"] = df["Category"].apply(target_days)
df["Target_Inventory"] = df["Daily_Demand"] * df["Target_Days"]

# =========================
# PRODUCTION QTY
# =========================

df["Production_Qty"] = np.where(
    df["Trigger"] == 1,
    df["Target_Inventory"] - df["Inventory"],
    0
)

df["Production_Qty"] = df["Production_Qty"].clip(lower=0)

# =========================
# HOURS REQUIRED
# =========================

df["Hours_Required"] = df["Production_Qty"] / df["Rate"]

# =========================
# PRIORITY
# =========================

priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
df["Priority"] = df["Category"].map(priority_map)

df = df.sort_values(by="Priority", ascending=False)

# =========================
# ASSIGN BEST MACHINE
# =========================

df = df.sort_values(by="Hours_Required")

df = df.drop_duplicates(subset=["Part"], keep="first")

# =========================
# MACHINE LOAD CHECK
# =========================

machine_load = df.groupby("Machine")["Hours_Required"].sum().reset_index()
machine_load["Overload"] = machine_load["Hours_Required"] - MACHINE_CAPACITY

# =========================
# HANDLE OVERLOAD
# =========================

for machine in machine_load["Machine"]:

    overload = machine_load.loc[machine_load["Machine"] == machine, "Overload"].values[0]

    if overload > 0:

        subset = df[df["Machine"] == machine]

        strangers = subset[subset["Category"]=="Stranger"]

        for idx in strangers.index:
            if overload <= 0:
                break
            overload -= df.loc[idx,"Hours_Required"]
            df.loc[idx,"Production_Qty"] = 0
            df.loc[idx,"Hours_Required"] = 0

        repeaters = subset[subset["Category"]=="Repeater"]

        for idx in repeaters.index:
            if overload <= 0:
                break
            overload -= df.loc[idx,"Hours_Required"]
            df.loc[idx,"Production_Qty"] = 0
            df.loc[idx,"Hours_Required"] = 0

# =========================
# FINAL OUTPUT
# =========================

df["Produce_Today"] = np.where(df["Production_Qty"]>0,"YES","NO")

df_final = df[[
    "Part",
    "Category",
    "Inventory",
    "Production_Qty",
    "Machine",
    "Hours_Required",
    "Produce_Today"
]]

df_final.to_excel("APS_Plan.xlsx", index=False)

print("APS Plan Generated")

ValueError: value_name (Machine) cannot match an element in the DataFrame columns.